<style>
.info {border-left:5px solid #2563eb;background:#eff6ff;padding:12px 16px;margin:12px 0}
.task {border-left:5px solid #d97706;background:#fffbeb;padding:12px 16px;margin:12px 0}
.success {border-left:5px solid #059669;background:#ecfdf5;padding:12px 16px;margin:12px 0}
.warning {border-left:5px solid #dc2626;background:#fef2f2;padding:12px 16px;margin:12px 0}
.definition {border-left:5px solid #7c3aed;background:#f5f3ff;padding:12px 16px;margin:12px 0}
</style>

        # Python for Data Science
        ## TW14 · Session 1 · Supervised Classification: From Baseline to Model Comparison

        **Course level:** developing beginner  
        **Prior learning:** TW13 feature roles, splitting, and preprocessing pipelines  
        **Duration:** approximately 90 minutes  
        **Method:** explain → predict → run → change → practise

        Supervised learning uses labelled examples to learn a relationship between predictors and an outcome. We compare a baseline, logistic regression, decision tree, and k-nearest neighbours on customer churn.

        **Student name:** __________________________ &nbsp;&nbsp; **Date:** __________________


        ## Learning outcomes

        By the end of this session, you should be able to:

        - define sample, feature, target, training, testing, fitting, and prediction
- create a stratified train/test split
- build leakage-safe preprocessing and model pipelines
- train logistic regression, decision tree, and k-NN classifiers
- interpret confusion matrices, precision, recall, F1, accuracy, and ROC AUC


## Lesson plan

| Part | Time | Activity |
|---|---:|---|
| Retrieval | 10 min | Recall earlier Python and data skills |
| New concepts | 30 min | Learn with short, explained examples |
| Guided analysis | 20 min | Build one complete example with the lecturer |
| Student coding | 25 min | Complete five marked tasks |
| Review | 5 min | Knowledge check and exit ticket |

<div class="info"><strong>How to work:</strong> read the explanation, predict the result, run the cell with <code>Shift + Enter</code>, then change one thing. Edit only cells marked <strong>Student task</strong> unless your lecturer says otherwise.</div>


### Load the labelled churn data

The target churn equals 1 for churn and 0 for retained. customer_id is an identifier, not a numeric predictor.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

_candidates = [
    Path.cwd() / "data",
    Path.cwd() / "Python_Data_Science_TW09_TW17" / "data",
]
DATA_DIR = next((path for path in _candidates if path.exists()), _candidates[0])
if not DATA_DIR.exists():
    raise FileNotFoundError("Data folder not found. Start JupyterLab from the extracted course folder.")
print("Data folder:", DATA_DIR.resolve())

churn_data = pd.read_csv(DATA_DIR / "customer_churn.csv")
print(churn_data.head())
print(churn_data["churn"].value_counts(normalize=True).sort_index())


## 1. Core terminology

- sample: one customer row;
- features X: inputs available before the outcome;
- target y: churn label;
- fit: learn model parameters from training examples;
- predict: produce labels for unseen rows;
- generalisation: performance on relevant unseen data.

A model is not useful merely because it fits training data.


### Separate identifiers, features, and target

The identifier is retained separately for tracing predictions but excluded from training.


In [ ]:
identifiers = churn_data["customer_id"].copy()
X = churn_data.drop(columns=["customer_id", "churn"])
y = churn_data["churn"]

print("X shape:", X.shape)
print("Target rate:", round(y.mean(), 3))


### Create a stratified holdout split

Stratification keeps the target proportion similar in train and test sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)
print("Train/test rows:", len(X_train), len(X_test))
print("Train/test churn rates:", round(y_train.mean(), 3), round(y_test.mean(), 3))


## 2. Preprocessing inside the model pipeline

Numerical features are imputed and scaled. Categories are imputed and one-hot encoded. Placing preprocessing inside the Pipeline ensures it is learned from training folds only.


### Define reusable preprocessing

The same preprocessor can be paired with several classifiers.


In [ ]:
numeric_columns = ["age", "tenure_months", "monthly_spend", "support_tickets"]
categorical_columns = ["contract_type", "autopay"]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipe, numeric_columns),
    ("categorical", categorical_pipe, categorical_columns),
])
print(preprocessor)


## 3. Establish a baseline

A baseline answers whether the model improves on a simple rule. The most-frequent baseline always predicts the majority class.


### Evaluate a majority-class baseline

Accuracy may look acceptable under imbalance even when no churner is identified, so we report several metrics.


In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, baseline_predictions), 3))
print("Precision:", round(precision_score(y_test, baseline_predictions, zero_division=0), 3))
print("Recall:", round(recall_score(y_test, baseline_predictions, zero_division=0), 3))
print("F1:", round(f1_score(y_test, baseline_predictions, zero_division=0), 3))


## 4. Train three classifiers

- Logistic regression models log-odds as a linear combination of features.
- Decision tree creates rule-based splits and can model non-linear interactions.
- k-NN predicts from nearby training observations and is sensitive to scaling.


### Fit logistic regression

The pipeline fits preprocessing and the classifier as one leakage-safe object.


In [ ]:
logistic_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
])
logistic_model.fit(X_train, y_train)
logistic_predictions = logistic_model.predict(X_test)
logistic_probabilities = logistic_model.predict_proba(X_test)[:, 1]
print("Logistic model fitted.")


### Fit decision tree and k-NN

A shallow tree reduces overfitting risk. k-NN uses an odd neighbour count to reduce ties.


In [ ]:
tree_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", DecisionTreeClassifier(max_depth=4, min_samples_leaf=8, random_state=42, class_weight="balanced")),
])
knn_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", KNeighborsClassifier(n_neighbors=9)),
])

tree_model.fit(X_train, y_train)
knn_model.fit(X_train, y_train)
print("Tree and k-NN fitted.")


## 5. Evaluate according to error costs

For churn:

- false negative: a churner is missed;
- false positive: a retained customer receives unnecessary intervention.

Recall prioritises finding churners; precision prioritises efficient interventions. F1 balances precision and recall. ROC AUC assesses ranking across thresholds.


### Create a metric table

Every model is evaluated on the same untouched test rows.


In [ ]:
fitted_models = {
    "Baseline": baseline,
    "Logistic regression": logistic_model,
    "Decision tree": tree_model,
    "k-NN": knn_model,
}
metric_rows = []
for name, model in fitted_models.items():
    predictions = model.predict(X_test)
    if hasattr(model, "predict_proba"):
        scores = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, scores)
    else:
        auc = np.nan
    metric_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1": f1_score(y_test, predictions, zero_division=0),
        "roc_auc": auc,
    })

model_comparison = pd.DataFrame(metric_rows).set_index("model")
print(model_comparison.round(3))


### Read a confusion matrix

Rows are actual classes and columns are predicted classes in scikit-learn.


In [ ]:
matrix = confusion_matrix(y_test, logistic_predictions)
confusion_table = pd.DataFrame(
    matrix,
    index=["Actual retained", "Actual churn"],
    columns=["Predicted retained", "Predicted churn"],
)
print(confusion_table)
print(classification_report(y_test, logistic_predictions, zero_division=0))


### Change the decision threshold

A lower threshold usually raises recall and lowers precision. The business chooses a threshold using error costs and capacity, not a default alone.


In [ ]:
threshold = 0.35
threshold_predictions = (logistic_probabilities >= threshold).astype(int)
threshold_report = {
    "threshold": threshold,
    "precision": precision_score(y_test, threshold_predictions, zero_division=0),
    "recall": recall_score(y_test, threshold_predictions, zero_division=0),
    "f1": f1_score(y_test, threshold_predictions, zero_division=0),
}
print(pd.Series(threshold_report).round(3))


## Guided model recommendation

Choose a model only after defining the priority metric, comparing against baseline, reviewing uncertainty, checking subgroup performance, and confirming deployment constraints.


### Select by a stated priority

This teaching example prioritises recall while requiring an improvement over the baseline F1.


In [ ]:
eligible = model_comparison.loc[model_comparison.index != "Baseline"]
recommended_name = eligible["recall"].idxmax()
recommended_metrics = eligible.loc[recommended_name]

print("Priority: identify as many churners as possible.")
print("Recommended candidate:", recommended_name)
print(recommended_metrics.round(3))
print("Next step: cross-validation and subgroup/error review.")


## Student coding lab

Use the provided dataset and keep the test set untouched until final evaluation.


        ### Student task 1: Create a valid split

        <div class="task"><strong>Your job:</strong> Separate identifier, X, and y; make a 70/30 stratified split with random_state 123; compare class rates.</div>

        **Check your work**

        - customer_id and churn are absent from X.
- Stratification is used.
- Training and test target rates are printed.


In [ ]:
# STUDENT TASK 1
task_X = pd.DataFrame()
task_y = pd.Series(dtype=int)
# Add the split and class-rate comparison.
print(task_X.shape, task_y.shape)


        ### Student task 2: Build the preprocessing pipeline

        <div class="task"><strong>Your job:</strong> Create numerical and categorical pipelines and combine them in a ColumnTransformer.</div>

        **Check your work**

        - Numeric values are imputed and scaled.
- Categories are imputed and one-hot encoded.
- Unknown categories will not crash transformation.


In [ ]:
# STUDENT TASK 2
task_preprocessor = None
print(task_preprocessor)


        ### Student task 3: Compare a baseline and logistic model

        <div class="task"><strong>Your job:</strong> Fit both models on training data and report accuracy, precision, recall, F1, and ROC AUC where available.</div>

        **Check your work**

        - Both models use the same split.
- Baseline performance is explicit.
- Metrics are calculated on test data only.


In [ ]:
# STUDENT TASK 3
task_results = pd.DataFrame()
print(task_results)


        ### Student task 4: Train tree and k-NN models

        <div class="task"><strong>Your job:</strong> Fit a constrained decision tree and a scaled k-NN pipeline; add their test metrics to the comparison.</div>

        **Check your work**

        - Preprocessing is inside each pipeline.
- Tree complexity is constrained.
- The comparison uses the same metric definitions.


In [ ]:
# STUDENT TASK 4
# Define and fit both pipelines.
tree_knn_results = pd.DataFrame()
print(tree_knn_results)


        ### Student task 5: Evaluate a decision threshold

        <div class="task"><strong>Your job:</strong> For logistic regression, compare thresholds 0.3, 0.5, and 0.7 using precision, recall, F1, and number predicted positive.</div>

        **Check your work**

        - Probabilities come from the test set.
- All three thresholds are compared.
- The recommendation refers to error costs and capacity.



**Optional extension:** Calculate subgroup recall by contract type and discuss unequal performance.


In [ ]:
# STUDENT TASK 5
threshold_results = []
# Loop over three thresholds and append dictionaries.
print(pd.DataFrame(threshold_results))


        ## Knowledge check

        Answer these without running new code first.

        1. Why exclude customer ID from predictors?
2. What does a majority baseline reveal?
3. How do false negatives differ from false positives?
4. Why does k-NN need scaling?
5. How does a threshold change precision and recall?


## Exit ticket and preparation

<div class="success"><strong>Exit ticket:</strong> write one idea you can explain, one operation you can code, and one question that remains.</div>

**Before the next session:** Session 2 adds LDA, Gaussian Naive Bayes, SVC, cross-validation, and a complete regression walkthrough.
